Análise da Dados 

In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast

In [37]:
negociaçoes = r"C:\Users\Orçamento\OneDrive - GRUPO RETEC\02. Engenharia\Dep. Orçamentos\POWERBI\AUTOMACAO RD\data\negociacoes_2025.xlsx"
df = pd.read_excel(negociaçoes)

In [38]:
print(df.head)

<bound method NDFrame.head of                            _id                        id  \
0     682229d4ab07780015785b59  682229d4ab07780015785b59   
1     682229c2462be9001bbbf184  682229c2462be9001bbbf184   
2     68221c31dd05c1001bfd953c  68221c31dd05c1001bfd953c   
3     6821f328f999bb0014de36f3  6821f328f999bb0014de36f3   
4     681e68e5a3d9a40018919799  681e68e5a3d9a40018919799   
...                        ...                       ...   
2874  677685aac7ad66001f69dbc5  677685aac7ad66001f69dbc5   
2875  6776855e2d52460014191210  6776855e2d52460014191210   
2876  6776855279d2c00017fdea61  6776855279d2c00017fdea61   
2877  677679be82e94600174715a2  677679be82e94600174715a2   
2878  677676ac2d5246001818fd9d  677676ac2d5246001818fd9d   

                                                   name  amount_montly  \
0     NCP 12295 - VENANCIO 2000 - HEXAG VESTIBULARES...            0.0   
1     NCP 12295 - VENANCIO 2000 - HEXAG VESTIBULARES...            0.0   
2                    NCP-12

In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2879 entries, 0 to 2878
Data columns (total 69 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   _id                                   2879 non-null   object 
 1   id                                    2879 non-null   object 
 2   name                                  2879 non-null   object 
 3   amount_montly                         2879 non-null   float64
 4   amount_unique                         2879 non-null   float64
 5   amount_total                          2879 non-null   float64
 6   prediction_date                       3 non-null      object 
 7   markup                                2879 non-null   object 
 8   last_activity_at                      1 non-null      object 
 9   interactions                          2879 non-null   int64  
 10  markup_last_activities                1 non-null      object 
 11  created_at       

In [40]:
df.describe()

,amount_montly,amount_unique,amount_total,interactions,rating,hold,win,organization.address_latitude,organization.address_longitude,stop_time_limit.expired,stop_time_limit.expired_days
count,2879.000000,2.879000e+03,2.879000e+03,2879.000000,2879.000000,0.0,1304.000000,1.000000,1.000000,297.000000,297.000000
mean,69.807527,3.761799e+04,3.768780e+04,1.352553,2.278222,NaN,0.909509,-16.708663,-49.235882,0.804714,30.138047
std,3745.614563,4.774967e+05,4.775059e+05,2.625525,1.829811,NaN,0.286994,NaN,NaN,0.397090,30.288441
min,0.000000,0.000000e+00,0.000000e+00,0.000000,1.000000,NaN,0.000000,-16.708663,-49.235882,0.000000,0.000000
25%,0.000000,0.000000e+00,0.000000e+00,0.000000,1.000000,NaN,1.000000,-16.708663,-49.235882,1.000000,2.000000
50%,0.000000,1.417200e+02,1.425300e+02,1.000000,1.000000,NaN,1.000000,-16.708663,-49.235882,1.000000,18.000000
75%,0.000000,2.072470e+03,2.076445e+03,1.000000,5.000000,NaN,1.000000,-16.708663,-49.235882,1.000000,52.000000
max,200975.870000,1.339870e+07,1.339870e+07,27.000000,5.000000,NaN,1.000000,-16.708663,-49.235882,1.000000,124.000000


In [41]:
df.columns

Index(['_id', 'id', 'name', 'amount_montly', 'amount_unique', 'amount_total',
       'prediction_date', 'markup', 'last_activity_at', 'interactions',
       'markup_last_activities', 'created_at', 'updated_at', 'rating',
       'markup_created', 'last_activity_content', 'user_changed', 'hold',
       'win', 'closed_at', 'contacts', 'deal_custom_fields', 'deal_products',
       'organization._id', 'organization.id', 'organization.name',
       'organization.address', 'organization.address_latitude',
       'organization.address_longitude', 'organization.user._id',
       'organization.user.id', 'organization.user.name',
       'organization.user.email', 'organization.organization_segments',
       'user._id', 'user.id', 'user.name', 'user.nickname', 'user.email',
       'deal_stage._id', 'deal_stage.id', 'deal_stage.name',
       'deal_stage.nickname', 'deal_stage.created_at', 'deal_stage.updated_at',
       'stop_time_limit.expiration_date_time', 'stop_time_limit.expired',
       'stop

In [56]:
def parse_custom_fields(s):
    if isinstance(s, str):
        try:
            return ast.literal_eval(s)
        except (ValueError, SyntaxError):
            return []
    elif isinstance(s, list):
        return s
    else:
        return []

df.loc[:, "deal_custom_fields"] = df["deal_custom_fields"].apply(parse_custom_fields)

# 3) Função que extrai o valor de “Fator”
def extrair_fator(campos):
    for campo in campos:
        cf = campo.get("custom_field", {})
        if cf.get("label", "").strip().lower() == "fator":
            return campo.get("value")
    return None

# 4) Aplique linha a linha e crie a coluna 'fator'
df.loc[:, "fator"] = df["deal_custom_fields"].apply(extrair_fator)

# 5) (Opcional) Converter string com vírgula para float
df.loc[:, "fator"] = (
    df["fator"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float, errors="ignore")
)

# 6) Filtrar só Bruno Crispim e Gabriel Bento e mostrar
vendedores = ["Bruno Crispim", "Gabriel  Bento"]
resultado = df[df["user.name"].isin(vendedores)][["id", "user.name", "fator"]]
print(resultado.head(20))

                           id       user.name   fator
65   681cb3ab978e2f00275fb27d   Bruno Crispim    0.85
66   681cae7e320c23001baa3e1d   Bruno Crispim    0.62
67   681cad029c91db001413c94a   Bruno Crispim       1
105  6818bf0a03c9f10014ecc56b   Bruno Crispim       1
106  6818bea73d2c6e0027bdcc88   Bruno Crispim       1
107  6818baf8ef82eb002c8ee57c   Bruno Crispim       1
118  6817d7e69bf42000143ae859  Gabriel  Bento    0.73
119  6817cc312f87ea002799dc84  Gabriel  Bento     0.6
120  6817cae7ae9fbd00148d8060  Gabriel  Bento       1
121  6817c9e49bf42000273ae212  Gabriel  Bento     0.8
122  6817c8d6473aea00174b1f1f  Gabriel  Bento       1
130  681510ed06910f0014cb7098   Bruno Crispim     0.7
146  6812921d25f0cc0025342649  Gabriel  Bento     0.6
147  6812920cbded940014949eaa  Gabriel  Bento     0.6
186  68113ad73bd702001493746d   Bruno Crispim     0.8
194  6811391ea12f2600210eda82   Bruno Crispim     0.8
239  6810e0be6f2584001be8d1b2  Gabriel  Bento    0.54
240  6810e0adaa4450001448da7

Análise, criando o dataframe com as colunas que queremos analisar

In [62]:
colunas_corrigidas = [
    "id", "name", "amount_total", "amount_unique", "markup",
    "created_at", "closed_at", "last_activity_at",
    "interactions", "win", "deal_stage.name", "user.id", "user.name",
    "deal_lost_reason.name",'fator'
]

df_analise = df[colunas_corrigidas].copy()

Transformando a coluna de Datas em tipo data

In [63]:
df_analise['created_at'] = pd.to_datetime(df_analise['created_at'], errors = 'coerce')
df_analise['closed_at'] = pd.to_datetime(df_analise['closed_at'], errors = 'coerce')
df_analise["last_activity_at"] = pd.to_datetime(df_analise["last_activity_at"], errors="coerce")

In [64]:
df_analise.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2879 entries, 0 to 2878
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype                    
---  ------                 --------------  -----                    
 0   id                     2879 non-null   object                   
 1   name                   2879 non-null   object                   
 2   amount_total           2879 non-null   float64                  
 3   amount_unique          2879 non-null   float64                  
 4   markup                 2879 non-null   object                   
 5   created_at             2879 non-null   datetime64[ns, UTC-03:00]
 6   closed_at              1304 non-null   datetime64[ns, UTC-03:00]
 7   last_activity_at       1 non-null      datetime64[ns, UTC-03:00]
 8   interactions           2879 non-null   int64                    
 9   win                    1304 non-null   float64                  
 10  deal_stage.name        2879 non-null   object   

In [65]:
df_analise.head()

,id,name,amount_total,amount_unique,markup,created_at,closed_at,last_activity_at,interactions,win,deal_stage.name,user.id,user.name,deal_lost_reason.name,fator
0,682229d4ab07780015785b59,NCP 12295 - VENANCIO 2000 - HEXAG VESTIBULARES...,0.00,0.00,future,2025-05-12 14:03:16.112000-03:00,NaT,NaT,0,NaN,Venda Ganha,651af6998076130019a8dda8,Luan Araújo,NaN,1
1,682229c2462be9001bbbf184,NCP 12295 - VENANCIO 2000 - HEXAG VESTIBULARES...,1256.89,1256.89,future,2025-05-12 14:02:58.767000-03:00,2025-05-12 14:03:15.952000-03:00,NaT,1,1.0,Venda Ganha,651af6998076130019a8dda8,Luan Araújo,NaN,1
2,68221c31dd05c1001bfd953c,NCP-12242 TCN 45 OBRA: RESIDENCIAL,0.00,0.00,future,2025-05-12 13:05:05.857000-03:00,NaT,NaT,0,NaN,Venda Ganha,651af77665445f0022476065,Wellisson Chaves,NaN,01
3,6821f328f999bb0014de36f3,NCP - REFRIAR REFRIGERACAO LTDA - ME,0.00,0.00,future,2025-05-12 10:10:00.555000-03:00,NaT,NaT,2,NaN,Orç. Pendente (Venda),655ca6a9d691d60021c81701,Marlon Souza,NaN,
4,681e68e5a3d9a40018919799,NCP-12229 TECNA OBRA: CME SARAH,44012.18,44012.18,future,2025-05-09 17:43:18.062000-03:00,NaT,NaT,0,NaN,Negociação,651af77665445f0022476065,Wellisson Chaves,NaN,01


In [66]:
# Converter campo win em booleano
df_analise["win"] = df_analise["win"].map({"true": True, "false": False})

# Recriar colunas auxiliares
df_analise["duracao_venda_dias"] = (df_analise["closed_at"] - df_analise["created_at"]).dt.days
df_analise["mes_criacao"] = df_analise["created_at"].dt.to_period("M")
#df_analise["status_final"] = df_analise["win"].map({True: "Ganho", False: "Perdido"})
#df_analise.loc[df_analise["closed_at"].isna(), "status_final"] = "Em Aberto"

C:\Users\Orçamento\AppData\Local\Temp\ipykernel_21828\2614631020.py:6: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_analise["mes_criacao"] = df_analise["created_at"].dt.to_period("M")


In [74]:
df_analise[::-10]

,id,name,amount_total,amount_unique,markup,created_at,closed_at,last_activity_at,interactions,win,deal_stage.name,user.id,user.name,deal_lost_reason.name,fator,duracao_venda_dias,mes_criacao
2878,677676ac2d5246001818fd9d,NCP 10317 - CASA VARANDA - ISRAEL GRADO ENGENH...,308.32,308.32,future,2025-01-02 08:21:16.306000-03:00,2024-12-31 01:00:00-03:00,NaT,1,NaN,Venda Ganha,651af6998076130019a8dda8,Luan Araújo,NaN,1,-3.0,2025-01
2868,6776c1f3d65038001edb2955,8098 - GEOLAB - GLEICIANE,28092.88,28092.88,future,2025-01-02 13:42:27.410000-03:00,NaT,NaT,1,NaN,Negociação,651af40601a5360011f74f6c,Rutemar Júnior,NaN,1,NaN,2025-01
2858,6777c299d650380018db8bed,NCP 10326 - SARAH LAGO NORTE - JICLIMAR REFRIG...,0.00,0.00,future,2025-01-03 07:57:29.274000-03:00,NaT,NaT,0,NaN,Venda Ganha,651af6998076130019a8dda8,Luan Araújo,NaN,1,NaN,2025-01
2848,6777ea816fbbb3001b6e5a94,NCP 10315 - FUNCEF - ALTRI CONSTRUTORA,8648.41,8648.41,future,2025-01-03 10:47:46.014000-03:00,2025-02-03 15:21:02.397000-03:00,NaT,11,NaN,Venda Perdida,651af6998076130019a8dda8,Luan Araújo,Relacionamento (Confiança e Parceria),1,31.0,2025-01
2838,6778280080810e001f34511d,NCP 10313 - CIVIL EMG - ARAUJO ABREU ENGENHARIA,1007.19,1007.19,future,2025-01-03 15:10:08.890000-03:00,2025-01-03 15:59:55.427000-03:00,NaT,1,NaN,Venda Ganha,651af6998076130019a8dda8,Luan Araújo,NaN,1,0.0,2025-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48,681ce3deff9c750027993495,PV - 14005JOAO BATISTA DA SILVA GRAMACHO 79452...,0.00,0.00,future,2025-05-08 14:03:26.330000-03:00,NaT,NaT,0,NaN,Venda Ganha,655ca6a9d691d60021c81701,Marlon Souza,NaN,1,NaN,2025-05
38,681d2f9babfac80027214f3f,12219 - RESIDENCIAL - RAPHAEL,0.00,0.00,future,2025-05-08 19:26:35.454000-03:00,NaT,NaT,0,NaN,Venda Ganha,65f878886cab6d0015e4690f,Iago Rangel,NaN,1,NaN,2025-05
28,681d32b9adebca00149c4ceb,12192 - ALGARVE - IDELMA,0.00,0.00,future,2025-05-08 19:39:53.280000-03:00,NaT,NaT,0,NaN,Venda Ganha,65f878886cab6d0015e4690f,Iago Rangel,NaN,1,NaN,2025-05
18,681d39d5fc6f4e001b916d8e,12286 - CAIXA ECONOMICA - MARUS,0.00,0.00,future,2025-05-08 20:10:13.146000-03:00,NaT,NaT,0,NaN,Venda Ganha,65f878886cab6d0015e4690f,Iago Rangel,NaN,1,NaN,2025-05


Estatísticas principais

In [86]:
df_analise['fator'] = pd.to_numeric(df_analise['fator'], errors='coerce')

In [89]:
resumo = {
    "Total de negociações": len(df_analise),
    "Contagem de Vendas Ganhas" : df_analise[df_analise["deal_stage.name"] == "Venda Ganha"].shape[0],
    "Contagem de Vendas Perdidas" : df_analise[df_analise["deal_stage.name"] == "Venda Perdida"].shape[0],
    "Contagem de Vendas Canceladas" : df_analise[df_analise["deal_stage.name"] == "Venda Cancelada"].shape[0],
    "Contagem de Vendas em aberto" : df_analise[df_analise["deal_stage.name"] == "Negociação"].shape[0],
    "Valor Total Ganhas": float(df_analise[df_analise["deal_stage.name"] == "Venda Ganha"]["amount_total"].sum()),
    "Valor Total Perdidas": float(df_analise[df_analise["deal_stage.name"] == "Venda Perdida"]["amount_total"].sum()),
    "Valor Total Canceladas": float(df_analise[df_analise["deal_stage.name"] == "Venda Cancelada"]["amount_total"].sum()),
    "Valor Total Não vendas": float(df_analise[df_analise["deal_stage.name"].isin(["Venda Perdida", "Venda Cancelada"])]["amount_total"].sum()),
    "Valor Total em Aberto": float(df_analise[df_analise["deal_stage.name"] == "Negociação"]["amount_total"].sum()),
    "Ticket Médio Ganho": float(df_analise[df_analise["deal_stage.name"] == "Venda Ganha"]["amount_total"].mean()),
    "Taxa de Desconto" : (float(df_analise['fator'].mean())),
    "Duração média (dias)": float(df_analise["duracao_venda_dias"].mean())
}

In [90]:
resumo

{'Total de negociações': 2879,
 'Contagem de Vendas Ganhas': 2398,
 'Contagem de Vendas Perdidas': 90,
 'Contagem de Vendas Canceladas': 27,
 'Contagem de Vendas em aberto': 128,
 'Valor Total Ganhas': 6287465.050000001,
 'Valor Total Perdidas': 3061888.3100000005,
 'Valor Total Canceladas': 1106928.4,
 'Valor Total Não vendas': 4168816.71,
 'Valor Total em Aberto': 10249327.62,
 'Ticket Médio Ganho': 2621.9620725604673,
 'Taxa de Desconto': 12.009243760922754,
 'Duração média (dias)': 3.5161042944785277}

Dados apenas dos vendedores da Representação (Bruno e Bento)

In [70]:
df_rep = df_analise[df_analise['user.name'].isin(['Bruno Crispim', 'Gabriel  Bento'])]
df_rep

,id,name,amount_total,amount_unique,markup,created_at,closed_at,last_activity_at,interactions,win,deal_stage.name,user.id,user.name,deal_lost_reason.name,fator,duracao_venda_dias,mes_criacao
65,681cb3ab978e2f00275fb27d,NCT 13989-25 - TECNA CONSTRUTORA - RESIDENCIAL...,0.00,0.0,future,2025-05-08 10:37:47.556000-03:00,NaT,NaT,0,NaN,Consulta Enviada,6557ecc0295062000f0ac40a,Bruno Crispim,NaN,0.85,NaN,2025-05
66,681cae7e320c23001baa3e1d,NCD 25050234-25 - FOX ENGENHARIA - BRB - 06.05...,200975.87,0.0,future,2025-05-08 10:15:42.148000-03:00,NaT,NaT,0,NaN,Consulta Enviada,6557ecc0295062000f0ac40a,Bruno Crispim,NaN,0.62,NaN,2025-05
67,681cad029c91db001413c94a,NCS 234974-25 - FOX ENGENHARIA - BRB - 06.05.2...,8962.60,8962.6,future,2025-05-08 10:09:22.366000-03:00,NaT,NaT,0,NaN,Consulta Enviada,6557ecc0295062000f0ac40a,Bruno Crispim,NaN,1,NaN,2025-05
105,6818bf0a03c9f10014ecc56b,NOP 109282-25 - UNIAO QUIMICA - VIRICAS I - 22...,7600.00,7600.0,future,2025-05-05 10:37:14.945000-03:00,NaT,NaT,0,NaN,Consulta Enviada,6557ecc0295062000f0ac40a,Bruno Crispim,NaN,1,NaN,2025-05
106,6818bea73d2c6e0027bdcc88,NCT-X 13965-25 - UNIAO QUIMICA - VIRICAS I - 2...,29000.00,29000.0,future,2025-05-05 10:35:35.514000-03:00,NaT,NaT,0,NaN,Consulta Enviada,6557ecc0295062000f0ac40a,Bruno Crispim,NaN,1,NaN,2025-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2814,677c3b699321820027e7d48c,NCI 3369-24 - CONNECTOR - MULTI CONSTRUTORA - ...,61000.00,61000.0,future,2025-01-06 17:22:01.459000-03:00,2025-01-30 11:45:59.249000-03:00,NaT,4,NaN,Venda Perdida,6557ecc0295062000f0ac40a,Bruno Crispim,Preço (perca para produto inferior),0.65,23.0,2025-01
2815,677c394b7aa09300141393a6,NCD 0050-24 - SOUSAR CLIMAENGENHARIA - SQS 303...,78958.70,78958.7,future,2025-01-06 17:12:59.268000-03:00,NaT,NaT,3,NaN,Orç. Enviar (Venda),6557ecc0295062000f0ac40a,Bruno Crispim,NaN,0.45,NaN,2025-01
2816,677c3882a38d3a0018b260fc,NCD 25010141-24 - NORTHEC - SESI - 03.01.2025 ...,0.00,0.0,future,2025-01-06 17:09:38.197000-03:00,NaT,NaT,0,NaN,Consulta Enviada,6557ecc0295062000f0ac40a,Bruno Crispim,NaN,0.45,NaN,2025-01
2818,677c368ac1cc7300169f18eb,NCD 0049 -REV2- RENOVAR ENGENHARIA - CAIXA ECO...,0.00,0.0,future,2025-01-06 17:01:14.939000-03:00,NaT,NaT,0,NaN,Venda Ganha,6557ecc0295062000f0ac40a,Bruno Crispim,NaN,0.39,NaN,2025-01


In [78]:
total_orc = len(df_rep["deal_stage.name"] == "Venda Ganha") + len(df_rep["deal_stage.name"] == "Venda Perdida") + len(df_rep["deal_stage.name"] == "Venda Cancelada")
taxa_conversão_rep = df_rep[df_rep["deal_stage.name"] == "Venda Ganha"].shape[0]/total_orc
df_rep['fator'] = df_rep['fator'].astype(float)

C:\Users\Orçamento\AppData\Local\Temp\ipykernel_21828\2486677785.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rep['fator'] = df_rep['fator'].astype(float)


In [81]:
resumo_rep = {
    "Total de negociações": len(df_rep),
    "Contagem de Vendas Ganhas" : df_rep[df_rep["deal_stage.name"] == "Venda Ganha"].shape[0],
    "Contagem de Vendas Perdidas" : df_rep[df_rep["deal_stage.name"] == "Venda Perdida"].shape[0],
    "Contagem de Vendas Canceladas" : df_rep[df_rep["deal_stage.name"] == "Venda Cancelada"].shape[0],
    "Contagem de Vendas em aberto" : df_rep[df_rep["deal_stage.name"] == "Negociação"].shape[0],
    "Valor Total Ganhas": float(df_rep[df_rep["deal_stage.name"] == "Venda Ganha"]["amount_total"].sum()),
    "Valor Total Perdidas": float(df_rep[df_rep["deal_stage.name"] == "Venda Perdida"]["amount_total"].sum()),
    "Valor Total Canceladas": float(df_rep[df_rep["deal_stage.name"] == "Venda Cancelada"]["amount_total"].sum()),
    "Valor Total Não vendas": float(df_rep[df_rep["deal_stage.name"].isin(["Venda Perdida", "Venda Cancelada"])]["amount_total"].sum()),
    "Valor Total em Aberto": float(df_rep[df_rep["deal_stage.name"] == "Negociação"]["amount_total"].sum()),
    "Ticket Médio Ganho": float(df_rep[df_rep["deal_stage.name"] == "Venda Ganha"]["amount_total"].mean()),
    "Taxa de Conversão ": float(taxa_conversão_rep),
    "Taxa de Desconto" : (1 - float(df_rep['fator'].mean())),
    "Duração média (dias)": float(df_rep["duracao_venda_dias"].mean())
}

In [82]:
resumo_rep

{'Total de negociações': 263,
 'Contagem de Vendas Ganhas': 100,
 'Contagem de Vendas Perdidas': 19,
 'Contagem de Vendas Canceladas': 6,
 'Contagem de Vendas em aberto': 32,
 'Valor Total Ganhas': 3160263.28,
 'Valor Total Perdidas': 2088878.7899999998,
 'Valor Total Canceladas': 574132.76,
 'Valor Total Não vendas': 2663011.5500000003,
 'Valor Total em Aberto': 8022858.470000001,
 'Ticket Médio Ganho': 31602.6328,
 'Taxa de Conversão ': 0.1267427122940431,
 'Taxa de Desconto': 0.15609733840304174,
 'Duração média (dias)': 7.739726027397261}